# Demo filter

A demonstration of the edge filter with controllable strength.

In [2]:
import math
import time
import tkinter as tk

import cv2
import numpy as np
from scipy.signal import convolve2d


# ---- Config ----
WINDOW_NAME = "Enric's CrazyCam"


KEY_ARROW_UP    = 82
KEY_ARROW_DOWN  = 84

INTENSITY_STEP  = 0.2
INTENSITY_MIN   = 0.1
INTENSITY_MAX   = 20.0

KEY_TOGGLE_EDGE = 'e'
KEY_TOGGLE_BLUR = 'b'
KEY_TOGGLE_WAVE = 'w'
KEY_QUIT        = 'q'


EFFECT_KEYS = {
    'edge': KEY_TOGGLE_EDGE,
    'blur': KEY_TOGGLE_BLUR,
    'wave': KEY_TOGGLE_WAVE,
}


root = tk.Tk()
screen_width  = root.winfo_screenwidth()
screen_height = root.winfo_screenheight()
root.destroy()


# Maintain aspect ratio and fill screen with black borders
def show_fullscreen_preserve_aspect(win_name, frame, screen_w, screen_h):
    h, w = frame.shape[:2]
    aspect_ratio = w / h
    target_ratio = screen_w / screen_h

    if aspect_ratio > target_ratio:
        new_w = screen_w
        new_h = int(screen_w / aspect_ratio)
    else:
        new_h = screen_h
        new_w = int(screen_h * aspect_ratio)

    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    result   = np.zeros((screen_h, screen_w, 3), dtype=np.uint8)
    y_offset = (screen_h - new_h) // 2
    x_offset = (screen_w - new_w) // 2
    result[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
    cv2.imshow(win_name, result)


# ---- Filters ----
def edge_filter(im, intensity):
    im_gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY).astype(np.float32)

    weight_edge_hor = np.array([[-1, 0, 1],
                                [-2, 0, 2],
                                [-1, 0, 1]])
    weight_edge_ver = np.array([[-1, -2, -1],
                                [ 0,  0,  0],
                                [ 1,  2,  1]])

    im_edge_ver = convolve2d(im_gray, weight_edge_hor, mode='same', boundary='symm')
    im_edge_hor = convolve2d(im_gray, weight_edge_ver, mode='same', boundary='symm')

    im_edge       = np.sqrt(im_edge_hor ** 2 + im_edge_ver ** 2)
    im_edge       = np.clip(im_edge * intensity, 0, 255).astype(np.uint8)
    edges_colored = cv2.cvtColor(im_edge, cv2.COLOR_GRAY2BGR)

    return cv2.addWeighted(im, 0.8, edges_colored, 0.6, 0)


def blur_filter(im, intensity):
    k = int(max(1, intensity * 3))
    if k % 2 == 0:
        k += 1
    return cv2.GaussianBlur(im, (k, k), 0)


def wave_filter(im, intensity):
    rows, cols, _ = im.shape
    im_out        = np.zeros_like(im)
    t             = time.time()

    for i in range(rows):
        shift_x   = int(6 * intensity * math.sin(2 * math.pi * i / 120 + t))
        im_out[i] = np.roll(im[i], shift_x, axis=0)

    return im_out


# ---- Overlay ----
def draw_overlay(im, effects):
    h, w, _ = im.shape
    overlay = np.zeros((40, w, 3), dtype=np.uint8)

    cv2.rectangle(overlay, (0, 0), (w, 40), (0, 0, 0), -1)

    text = " | ".join(
        [f"{k.upper()}:{'ON' if v['on'] else 'OFF'} ({v['intensity']:.1f})"
         for k, v in effects.items()]
    )

    cv2.putText(overlay, text, (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

    return np.vstack([overlay, im])


# ---- Main ----
cv2.startWindowThread()
camera = cv2.VideoCapture(0)

effects = {
    'edge': {'on': False, 'intensity': 1.0},
    'blur': {'on': False, 'intensity': 1.0},
    'wave': {'on': False, 'intensity': 1.0},
}

last_active = None

print("Controls:")
print(f" {KEY_TOGGLE_EDGE.upper()} - toggle Edge filter")
print(f" {KEY_TOGGLE_BLUR.upper()} - toggle Blur filter")
print(f" {KEY_TOGGLE_WAVE.upper()} - toggle Wave filter")
print(" Up / Down arrows - adjust intensity of last toggled filter")
print(f" {KEY_QUIT.upper()} - quit")

cv2.namedWindow(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN)
cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

key_to_effect = {
    ord(EFFECT_KEYS['edge']): 'edge',
    ord(EFFECT_KEYS['blur']): 'blur',
    ord(EFFECT_KEYS['wave']): 'wave',
}

try:
    while True:
        ret, im = camera.read()
        if not ret:
            break

        if effects['edge']['on']:
            im = edge_filter(im, effects['edge']['intensity'])
        if effects['blur']['on']:
            im = blur_filter(im, effects['blur']['intensity'])
        if effects['wave']['on']:
            im = wave_filter(im, effects['wave']['intensity'])

        im_display = draw_overlay(im, effects)
        show_fullscreen_preserve_aspect(WINDOW_NAME, im_display, screen_width, screen_height)

        key = cv2.waitKey(1) & 0xFF

        if key in key_to_effect:
            effect_name                 = key_to_effect[key]
            effects[effect_name]['on']  = not effects[effect_name]['on']
            last_active                 = effect_name
            print(f'{effect_name.capitalize()} {"ON" if effects[effect_name]["on"] else "OFF"}')

        elif key == KEY_ARROW_UP:
            if last_active:
                effects[last_active]['intensity'] = min(
                    INTENSITY_MAX,
                    effects[last_active]['intensity'] + INTENSITY_STEP
                )
                print(f'{last_active} intensity: {effects[last_active]["intensity"]:.1f}')

        elif key == KEY_ARROW_DOWN:
            if last_active:
                effects[last_active]['intensity'] = max(
                    INTENSITY_MIN,
                    effects[last_active]['intensity'] - INTENSITY_STEP
                )
                print(f'{last_active} intensity: {effects[last_active]["intensity"]:.1f}')

        elif key == ord(KEY_QUIT):
            break

finally:
    camera.release()
    cv2.destroyAllWindows()
    for _ in range(4):
        cv2.waitKey(1)

Controls:
 E - toggle Edge filter
 B - toggle Blur filter
 W - toggle Wave filter
 Up / Down arrows - adjust intensity of last toggled filter
 Q - quit


QObject::moveToThread: Current thread (0x6642320) is not the object's thread (0x706dfd0).
Cannot move to target thread (0x6642320)

QObject::moveToThread: Current thread (0x6642320) is not the object's thread (0x706dfd0).
Cannot move to target thread (0x6642320)

QObject::moveToThread: Current thread (0x6642320) is not the object's thread (0x706dfd0).
Cannot move to target thread (0x6642320)

QObject::moveToThread: Current thread (0x6642320) is not the object's thread (0x706dfd0).
Cannot move to target thread (0x6642320)

QObject::moveToThread: Current thread (0x6642320) is not the object's thread (0x706dfd0).
Cannot move to target thread (0x6642320)

QObject::moveToThread: Current thread (0x6642320) is not the object's thread (0x706dfd0).
Cannot move to target thread (0x6642320)

QObject::moveToThread: Current thread (0x6642320) is not the object's thread (0x706dfd0).
Cannot move to target thread (0x6642320)

QObject::moveToThread: Current thread (0x6642320) is not the object's thread

Edge ON
Blur ON
blur intensity: 1.2
blur intensity: 1.4
blur intensity: 1.6
blur intensity: 1.8
blur intensity: 2.0
blur intensity: 2.2
blur intensity: 2.4
blur intensity: 2.6
blur intensity: 2.8
blur intensity: 3.0
blur intensity: 3.2
blur intensity: 3.4
blur intensity: 3.6
blur intensity: 3.8
blur intensity: 4.0
blur intensity: 4.2
blur intensity: 4.4
blur intensity: 4.6
blur intensity: 4.8
blur intensity: 5.0
blur intensity: 5.2
blur intensity: 5.4
blur intensity: 5.6
blur intensity: 5.8
blur intensity: 6.0
blur intensity: 6.2
blur intensity: 6.4
blur intensity: 6.6
blur intensity: 6.8
blur intensity: 7.0
blur intensity: 7.2
blur intensity: 7.4
blur intensity: 7.6
blur intensity: 7.8
blur intensity: 8.0
blur intensity: 8.2
blur intensity: 8.4
blur intensity: 8.6
blur intensity: 8.8
blur intensity: 9.0
blur intensity: 9.2
blur intensity: 9.4
blur intensity: 9.2
blur intensity: 9.0
blur intensity: 8.8
blur intensity: 8.6
blur intensity: 8.4
blur intensity: 8.2
blur intensity: 8.0
blur